# Gaia LOD pyramid

Free Colab. One Gaia shard at a time, plain NumPy, hard 2 GB RSS cap.
No GitHub token and no DR3 download. Output is a new Drive folder.

`MODE = "PILOT"` reads the 20 shards listed below (267.9 MB, 4,321,186 records).
`MODE = "FULL"` reads every `GaiaSource_*.bin` under `BinFiles_2025` and `BinFiles_2025_Continued`.


In [ ]:
import os, sys, subprocess

BRANCH = "cursor/staris-pyramid-d6a4"
REPO = "https://github.com/caradmico/space-apps-groton.git"
ROOT = "/content/staris"
if os.path.exists("/content") and not os.path.exists(os.path.join(ROOT, "tools/staris-pipeline/pyramid.py")):
    subprocess.check_call(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, ROOT])
if os.path.exists(ROOT):
    os.chdir(ROOT)
    sys.path.insert(0, os.path.join(ROOT, "tools/staris-pipeline"))
else:
    # Running from a checkout of this repo, not Colab.
    ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", "..")) if not os.path.exists("tools/staris-pipeline/pyramid.py") else os.getcwd()
    sys.path.insert(0, os.path.join(ROOT, "tools/staris-pipeline"))
import pyramid
print("pyramid", pyramid.__file__)


In [ ]:
# Colab only. A local checkout skips this.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive"
except ImportError:
    DRIVE = None
    print("not Colab; set shard_paths in the run cell to test locally")


## Settings

`WRITE_DEEP` is the R2 flag. Leave it `False` until a bucket exists: orders 7–12 are counted and not written.
The default adapter is `drive_bin_dr1`. `esa_csv` only reads a csv.gz path you already have
(`source_id`, `ra`, `dec`, `parallax`, `phot_g_mean_mag`). It does not download DR3.
The 14 missing DR1 files, if you fetch them yourself, are at
`https://cdn.gea.esac.esa.int/Gaia/gdr1/gaia_source/csv/`.


In [ ]:
MODE = "PILOT"          # or "FULL"
ADAPTER = "drive_bin_dr1"  # or "esa_csv"
WRITE_DEEP = False         # R2 flag: write orders 7..12
RSS_CAP = 2 * 1024**3
PILOT = pyramid.PILOT_SHARDS

# Leave STARIS as None to search Drive. Set it if the search misses.
STARIS = None
OUTPUT = os.path.join(DRIVE, "StarIS_lod_v2") if DRIVE else "/tmp/staris-lod-v2"


In [ ]:
def find_staris(drive_root):
    direct = os.path.join(drive_root, "StarIS")
    if os.path.isdir(os.path.join(direct, "BinFiles_2025")):
        return direct
    for dirpath, dirnames, _files in os.walk(drive_root):
        if os.path.basename(dirpath) == "StarIS" and "BinFiles_2025" in dirnames:
            return dirpath
        # Don't walk the output folder or other people's trees forever.
        if os.path.basename(dirpath) in ("StarIS_lod_v2", ".Trash"):
            dirnames[:] = []
    return None

if DRIVE:
    if STARIS is None:
        STARIS = find_staris(DRIVE)
    if not STARIS:
        raise SystemExit("Set STARIS to the Drive folder that contains BinFiles_2025")
    BIN = os.path.join(STARIS, "BinFiles_2025")
    CONT = os.path.join(BIN, "BinFiles_2025_Continued")
    INPUT_DIRS = [path for path in (BIN, CONT) if os.path.isdir(path)]
    print("StarIS", STARIS)
    print("inputs", INPUT_DIRS)
else:
    INPUT_DIRS = []
print("output", OUTPUT)
print("pilot shards", len(PILOT))


In [ ]:
config = {
    "mode": MODE,
    "adapter": ADAPTER,
    "pilot_shards": PILOT,
    "input_dirs": INPUT_DIRS,
    "output_dir": OUTPUT,
    "rss_cap": RSS_CAP,
    "write_deep": WRITE_DEEP,
}
# Resume is automatic: manifest.json in OUTPUT remembers finished shards and closed cells.
manifest = pyramid.run_pipeline(config)
print("pass", manifest.get("pass"), "aborted", manifest.get("aborted"))
print("verify", manifest.get("verify"))
print("coarse", manifest.get("coarse_counts"))
print("deep", manifest.get("deep_counts"))
